## openai_api_call 开发实战

```bash
curl -H "Content-Type: application/json" \
     -H "Authorization: Bearer sk-xxxxxxxxxx" \
     -X POST \
     -d '{"model": "gpt-3.5-turbo", "messages": [{"role": "user", "content":
"你好"}]}' \
     https://api.openai.com/v1/chat/completions
```

In [ ]:
!pip install openai_api_call --upgrade

## 基本使用

### 对话参数

API 文档地址：https://platform.openai.com/docs/guides/chat/introduction

messages 支持三种角色：

 - `system` 有助于设置 AI 助手的行为
 - `user` 用于指导 AI，由用户或者开发者直接设置
 - `assisant` 存储之前的回复内容，也可以由开发者编写，帮助指导 AI

可选参数：
 - `model`：默认 `gpt-turbo-3.5`
 - `temperature`：简单来说，`temperature` 的参数值越小，模型就会返回越确定的一个结果。如果调高该参数值，大语言模型可能会返回更随机的结果，也就是说这可能会**带来更多样化或更具创造性的产出**。我们目前也在增加其他可能 token 的权重。在实际应用方面，对于质量保障 （QA） 等任务，我们可以设置更低的 `temperature` 值，以促使模型基于事实返回更真实和简洁的结果。对于诗歌生成或其他创造性任务，你可以适当调高 `temperature` 参数值。
 - `Top_p`:同样，使用 `top_p` （与 `temperature`一起称为核采样的技术），可以用来控制模型返回**结果的真实性**。如果你需要准确和事实的答案，就把参数值调低。如果你想要更多样化的答案，就把参数值调高一些。

建议改变其中一个参数，不用两个都调整。请记住最终生成的结果可能会和使用的大语言模型的版本而异。

### 基本用法

In [ ]:
from openai_api_call import *

## 基础调用
chat = Chat() # 新建空对话
chat.system("You are a helpful assistant")
chat.user("Print 'hello world' using Python")
resp = chat.getresponse()
print(resp) # 等同于 print(chat[-1])

In [ ]:
## 打印对话历史
chat.print_log()

In [ ]:
chat.show_usage_status()

### 批量处理数据

示例：
- MathPrompter: Mathematical Reasoning using Large Language Models(2023.03)
- https://arxiv.org/pdf/2303.05398.pdf
- zero_shot_cot/MultiArith

In [ ]:
## 使用 checkpoint 功能
import json
with open("MultiArith.json", "r") as f:
    dataset = json.load(f)

In [ ]:
d0 = dataset[0]
d0

In [ ]:
que = d0['sQuestion']
chat = Chat()
chat.system('You are a skilled math-problem solver. Response using Json, Example: {"answer":23}')
prefix = "Please write the answer to the question without showing the process.\n"
chat.user(prefix + que)
resp = chat.getresponse()
chat.print_log()

In [ ]:
def data2chat(data):
    que = data['sQuestion']
    chat = Chat()
    chat.system('You are a skilled math-problem solver. Response using Json, Example: {"answer":23}')
    prefix = "Please write the answer to the question without showing the process.\n"
    chat.user(prefix + que)
    resp = chat.getresponse(max_requests=3, timeout=20)
    print("done")
    return chat
#     chat.print_log()

In [ ]:
# 更多参数
?chat.getresponse

In [ ]:
chkpath = "math.log"
chats = process_chats(dataset[:5], data2chat, chkpath) #, clearfile=True)

In [ ]:
process_chats(dataset[:5], data2chat, chkpath, last_message_only=True)

In [ ]:
load_chats(chkpath, last_message_only=True)

In [ ]:
results = load_chats(chkpath, last_message_only=True)
for res, data in zip(results, dataset):
    print(data['lSolutions'])
    print(json.loads(res)['answer'])
    print("-"*10)

## 配置密钥和代理

设置环境变量

- ubuntu/mac 在系统的 `~.bashrc` 或 `~/.zshrc` 下设置
- win 安装 git 后在 `~/.bashrc` 下设置

```bash
# 配置密钥
export OPENAI_API_KEY=sk-xxx

# 配置代理链接
export OPENAI_BASE_URL=xxx

# 配置网络代理
export http_proxy=xxx
export https_proxy=xxx
```

In [ ]:
from openai_api_call import *
import openai_api_call

# 设置 API
# openai_api_call.api_key = "sk-xxx"

# 设置代理链接
# openai_api_call.base_url = "api.example.com"

# 设置服务器代理
proxy_on(http="127.0.0.1:7890")
proxy_status()
proxy_off()

## 开发指南

1. 文件结构
2. 函数注释，类型标注 | Copilot
3. 自动测试，部署，发行